<a href="https://colab.research.google.com/github/Ganzadidier/LLMs-Fine-Tuning/blob/Master/MedAssist_Quick_Demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏥 MedAssist — Quick Demo Notebook

This notebook **downloads the fine-tuned MedAssist model from Hugging Face** and launches an interactive Gradio chatbot — no training required.

**Run all cells top to bottom** (`Runtime → Run all`) and click the public link that appears at the end.

---
| | |
|---|---|
| **Base model** | Qwen/Qwen2.5-1.5B-Instruct |
| **Adapter** | LoRA (r=64, trained on 2,000 medical Q&A pairs) |
| **Dataset** | medalpaca/medical_meadow_medical_flashcards |
| **Runtime needed** | GPU (T4 or better) — *Runtime → Change runtime type → T4 GPU* |


## Step 1 — Install dependencies

In [1]:
# Install required packages
!pip install -q transformers peft accelerate bitsandbytes gradio huggingface_hub
print("✅ Dependencies installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.9 MB/s eta 0:00:00
✅ Dependencies installed


In [9]:
!pip install gradio==5.50.0 -q

## Step 2 — Download model from Hugging Face

In [2]:
from huggingface_hub import snapshot_download
import os

HF_REPO      = "dganza/medassist-qwen25-lora"   # ← update if needed
LOCAL_DIR    = "./medassist_best_model"

if os.path.exists(f"{LOCAL_DIR}/adapter_model.safetensors"):
    print(f"✅ Model already downloaded at {LOCAL_DIR} — skipping.")
else:
    print(f"⏳ Downloading from {HF_REPO} ...")
    snapshot_download(repo_id=HF_REPO, local_dir=LOCAL_DIR)
    print(f"✅ Downloaded to {LOCAL_DIR}")

⏳ Downloading from dganza/medassist-qwen25-lora ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

✅ Downloaded to ./medassist_best_model


## Step 3 — Load model

In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
import torch, json

print(f"CUDA available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU            : {torch.cuda.get_device_name(0)}")
else:
    print("⚠️  No GPU — responses will be slow. Change runtime to T4 GPU.")

BEST_MODEL_DIR = "./medassist_best_model"

with open(f"{BEST_MODEL_DIR}/model_metadata.json") as f:
    meta = json.load(f)
print(f"Best experiment : {meta['best_experiment']}")

tokenizer = AutoTokenizer.from_pretrained(BEST_MODEL_DIR, trust_remote_code=True)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = "left"

try:
    import bitsandbytes
    bnb_cfg = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
    )
    base_model = AutoModelForCausalLM.from_pretrained(
        meta["base_model"],
        quantization_config=bnb_cfg,
        device_map="auto",
        trust_remote_code=True,
        low_cpu_mem_usage=True,
    )
    print("⚙️  Loaded with 4-bit quantization")
except Exception:
    base_model = AutoModelForCausalLM.from_pretrained(
        meta["base_model"],
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True,
        low_cpu_mem_usage=True,
    )
    print("⚙️  Loaded with float16 (no quantization)")

ft_model = PeftModel.from_pretrained(base_model, BEST_MODEL_DIR)
ft_model.eval()
print(f"\n✅ ft_model ready — device: {next(ft_model.parameters()).device}")

CUDA available : True
GPU            : Tesla T4
Best experiment : Exp4_r64_lr2e4_2ep


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

⚙️  Loaded with 4-bit quantization

✅ ft_model ready — device: cuda:0


## Step 4 — Define chat function

In [4]:
import torch

SYSTEM_PROMPT = (
    "You are MedAssist, a knowledgeable medical education assistant. "
    "Provide clear, accurate, and concise answers to medical questions. "
    "Always remind users that your answers are for educational purposes only."
)

def chat(model, question, context=""):
    user_content = (
        f"Context: {context}\n\nQuestion: {question}"
        if context.strip() else question
    )
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_content},
    ]
    input_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(
        input_text, return_tensors="pt", truncation=True, max_length=512
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    generated = outputs[0][inputs["input_ids"].shape[1]:]
    response  = tokenizer.decode(generated, skip_special_tokens=True).strip()
    return response or "I wasn't able to generate a response. Please rephrase your question."

print("✅ chat() ready")

✅ chat() ready


## Step 5 — Launch Gradio app

In [11]:
import gradio as gr
print(f"Gradio version: {gr.__version__}")

# ── ────────────────────────────────────────────────────────────────────────────
#  CONFIG
# ── ────────────────────────────────────────────────────────────────────────────
QUICK_PROMPTS = [
    "💊 How does metformin work?",
    "🫀 Which nerve controls facial expression?",
    "🦠 Symptoms of type 2 diabetes?",
    "🔬 How to diagnose iron deficiency anemia?",
]

CSS = """
@import url('https://fonts.googleapis.com/css2?family=Plus+Jakarta+Sans:wght@300;400;500;600;700&display=swap');

* { font-family: 'Plus Jakarta Sans', sans-serif !important; }

/* ── Page ── */
.gradio-container {
    background: #0a0f0d !important;
    min-height: 100vh;
}
footer { display: none !important; }

/* ── Welcome ── */
.welcome-text h1 {
    color: #ffffff !important;
    font-size: 2em !important;
    font-weight: 700 !important;
    text-align: center !important;
    margin: 28px 0 4px 0 !important;
    letter-spacing: -0.03em !important;
}
.welcome-text p {
    color: #4b7a5e !important;
    text-align: center !important;
    font-size: 0.92em !important;
    margin-bottom: 20px !important;
}

/* ── Quick chips ── */
.quick-btn button {
    background: #111a14 !important;
    border: 1px solid #1e3325 !important;
    border-radius: 999px !important;
    color: #7ecfa0 !important;
    font-size: 0.82em !important;
    font-weight: 500 !important;
    padding: 7px 16px !important;
    transition: all 0.18s !important;
}
.quick-btn button:hover {
    background: #162b1e !important;
    border-color: #2dbe6c !important;
    color: #ffffff !important;
    transform: translateY(-2px) !important;
    box-shadow: 0 4px 14px rgba(45,190,108,0.2) !important;
}

/* ── Input bar ── */
.input-row textarea {
    background: #111a14 !important;
    border: 1.5px solid #1e3325 !important;
    border-radius: 16px !important;
    color: #e2f0e8 !important;
    font-size: 0.95em !important;
    padding: 14px 18px !important;
    resize: none !important;
    transition: border-color 0.2s, box-shadow 0.2s !important;
}
.input-row textarea:focus {
    border-color: #2dbe6c !important;
    box-shadow: 0 0 0 3px rgba(45,190,108,0.12) !important;
}
.input-row textarea::placeholder { color: #2d4a38 !important; }

/* ── Send button ── */
.send-btn button {
    background: linear-gradient(135deg, #2dbe6c 0%, #1a9e55 100%) !important;
    border: none !important;
    border-radius: 14px !important;
    color: #ffffff !important;
    font-weight: 600 !important;
    font-size: 0.95em !important;
    letter-spacing: 0.01em !important;
    box-shadow: 0 4px 16px rgba(45,190,108,0.3) !important;
    transition: all 0.2s !important;
}
.send-btn button:hover {
    transform: translateY(-2px) !important;
    box-shadow: 0 8px 24px rgba(45,190,108,0.45) !important;
}

/* ── Clear button ── */
.clear-btn button {
    background: transparent !important;
    border: 1.5px solid #1e3325 !important;
    border-radius: 14px !important;
    color: #4b7a5e !important;
    transition: all 0.2s !important;
}
.clear-btn button:hover {
    border-color: #f87171 !important;
    color: #f87171 !important;
}

/* ── Chatbot window ── */
.chat-area .wrap {
    background: transparent !important;
    border: none !important;
    box-shadow: none !important;
}
.chat-area {
    margin-top: 18px;
}

/* ── Message bubbles override ── */
.message.user-message div, .message.user-message p {
    background: #162b1e !important;
    color: #d4f0e0 !important;
    border-radius: 18px 18px 4px 18px !important;
    border: 1px solid #1e3325 !important;
}
.message.bot-message div, .message.bot-message p {
    background: #0d1710 !important;
    color: #c8e6d4 !important;
    border-radius: 18px 18px 18px 4px !important;
    border: 1px solid #1a2e20 !important;
}

/* ── Disclaimer ── */
.disclaimer p {
    color: #253d2e !important;
    font-size: 0.74em !important;
    text-align: center !important;
    margin-top: 10px !important;
}
"""

# ── ────────────────────────────────────────────────────────────────────────────
#  LOGIC
# ── ────────────────────────────────────────────────────────────────────────────
def respond(question, history):
    if not question.strip():
        return history, ""
    try:
        answer = chat(ft_model, question.strip(), "")
        answer += "\n\n---\n*⚕️ For educational purposes only. Always consult a healthcare professional.*"
    except Exception as e:
        answer = f"⚠️ **Error:** {e}\n\nMake sure the model loading cell was run first."
    history.append({"role": "user",      "content": question.strip()})
    history.append({"role": "assistant", "content": answer})
    return history, ""

def clear_chat():
    return [], ""

def quick_ask(prompt, history):
    # strip leading emoji + space
    clean = " ".join(prompt.split(" ")[1:])
    return respond(clean, history)

# ── ────────────────────────────────────────────────────────────────────────────
#  INTERFACE
# ── ────────────────────────────────────────────────────────────────────────────
with gr.Blocks(title="MedAssist", css=CSS) as demo:

    history_state = gr.State([])

    # ── Welcome ──────────────────────────────────────────────────────────────
    gr.Markdown(
        "# 🌿 Hey there. What would you like to learn?\nI'm MedAssist — your medical education companion.",
        elem_classes=["welcome-text"],
    )

    # ── Quick chips ──────────────────────────────────────────────────────────
    with gr.Row():
        quick_btns = [
            gr.Button(p, scale=1, elem_classes=["quick-btn"])
            for p in QUICK_PROMPTS
        ]

    # ── INPUT on top ─────────────────────────────────────────────────────────
    with gr.Row(elem_classes=["input-row"]):
        question_box = gr.Textbox(
            placeholder="Ask a medical question...   (Shift+Enter to send)",
            lines=2,
            max_lines=5,
            label="",
            scale=6,
        )
        with gr.Column(scale=1, min_width=100):
            send_btn  = gr.Button("Send ↩",  elem_classes=["send-btn"])
            clear_btn = gr.Button("Clear",   elem_classes=["clear-btn"])

    # ── CHAT HISTORY below ───────────────────────────────────────────────────
    chatbot = gr.Chatbot(
        height=420,
        type="messages",
        elem_classes=["chat-area"],
        placeholder=(
            "### 👋 Ask me anything about medicine!\n\n"
            "Type a question above or click one of the quick prompts. "
            "Your answers will appear here."
        ),
    )

    # ── Disclaimer ───────────────────────────────────────────────────────────
    gr.Markdown(
        "⚠️ Educational use only · Not a substitute for professional medical advice · Always consult a licensed clinician",
        elem_classes=["disclaimer"],
    )

    # ── Wiring ───────────────────────────────────────────────────────────────
    send_btn.click(
        fn=respond,
        inputs=[question_box, history_state],
        outputs=[chatbot, question_box],
    )
    question_box.submit(
        fn=respond,
        inputs=[question_box, history_state],
        outputs=[chatbot, question_box],
    )
    clear_btn.click(
        fn=clear_chat,
        outputs=[chatbot, question_box],
    )
    for btn in quick_btns:
        btn.click(
            fn=quick_ask,
            inputs=[btn, history_state],
            outputs=[chatbot, question_box],
        )

demo.launch(share=True, debug=False)

Gradio version: 5.50.0


/tmp/ipython-input-936637470.py:167: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(title="MedAssist", css=CSS) as demo:
/tmp/ipython-input-936637470.py:198: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9477c0f4b45ec4199d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
